<a href="https://colab.research.google.com/github/ChSurya/Cloud-Data-Pipeline-by-using-SQL/blob/main/Joins_SQL_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

client = bigquery.Client('scaler-sql-476312')

# **JOINS**

In [2]:
# How do we know the primary key basically it will mention in ER Diagram if they didn't there is a way we can find the primary key
query = """
SELECT count(*) as no_of_rows , count(customer_id) as unique
FROM `farmers_market.customer_purchases`
"""
df = client.query(query).to_dataframe()
df

# Since they both are same so customer_id is a Primary key for customer table

,no_of_rows,unique
0,1003,1003


In [3]:
 # Question - – Get a list of customers' zip codes for customers who made a purchase on 2019-04-06.
query = """
SELECT cu.customer_zip
FROM `farmers_market.customer` cu
LEFT JOIN `farmers_market.customer_purchases` cp
  ON cu.customer_id = cp.customer_id
WHERE cp.market_date = "2019-04-06"
"""

df = client.query(query).to_dataframe()
df

,customer_zip
0,22801
1,22801
2,22801
3,22801
4,22801
5,22821
6,22821
7,22821


In [4]:
 # Question - – Get the list of customers who have never made any purchase
query = """
SELECT cu.customer_id,cu.customer_first_name
FROM `farmers_market.customer` cu
LEFT JOIN `farmers_market.customer_purchases` cp
  USING (customer_id)
WHERE cp.customer_id IS NULL
"""

df = client.query(query).to_dataframe()
df

,customer_id,customer_first_name
0,56,Rohit
1,55,James


In [5]:
 # Question - – There are customers who's information is deleted from the customers table who had previously made the purchase.
query = """
SELECT cp.customer_id
FROM `farmers_market.customer` cu
RIGHT JOIN `farmers_market.customer_purchases` cp
  ON cu.customer_id = cp.customer_id
WHERE cu.customer_id IS NULL
"""

df = client.query(query).to_dataframe()
df

,customer_id
0,57
1,58
2,59


In [6]:
# Question -- Find out the customers who are either new to the market or have deleted their account from the market.
query = """
SELECT cp.customer_id AS deleted_cust, cu.customer_id AS new_cust
FROM `farmers_market.customer` cu
FULL JOIN `farmers_market.customer_purchases` cp
  ON cu.customer_id = cp.customer_id
WHERE cu.customer_id IS NULL OR cp.customer_id IS NULL
"""
df = client.query(query).to_dataframe()
df

,deleted_cust,new_cust
0,<NA>,56
1,<NA>,55
2,57,<NA>
3,58,<NA>
4,59,<NA>


# **UNION DISTINCT FOR BIQ QUERY / UNION FOR SQL**

In Below code we find the who are new customers and who are the deleted customers for that we use **UNION DISTINCT for Bigquery / UNION for SQL**

# **"AS Column_name"**

In Final result we got the customers who are new and who are deleted but we don't know which customer is new or deleted so we use **AS COL_NAME**

In below question we use as and give the col_name as type so we can identify which customer is new or deleted

In [7]:
#Question -- Find out the customers who are either new to the market or have deleted their account from the market. [Same above Question but we will find the results by using UNION instead of full join]
query = """
(
  SELECT cu.customer_id , "New_Customer" AS type
  FROM `farmers_market.customer` cu
  LEFT JOIN `farmers_market.customer_purchases` cp
    ON cu.customer_id = cp.customer_id
  WHERE cp.customer_id IS NULL
)
UNION DISTINCT
(
  SELECT cp.customer_id,"Deleted_Customer" AS type
  FROM `farmers_market.customer` cu
  RIGHT JOIN `farmers_market.customer_purchases` cp
    ON cu.customer_id = cp.customer_id
  WHERE cu.customer_id IS NULL
)

"""
df = client.query(query).to_dataframe()
df

,customer_id,type
0,56,New_Customer
1,55,New_Customer
2,57,Deleted_Customer
3,58,Deleted_Customer
4,59,Deleted_Customer
